
# 🇰🇷 Korean Morphology — *Tokens Only* (Jupyter)
**형태소 분석(토큰화)만 수행**하는 미니 노트북입니다.
- `Okt.pos()`로 형태소 태깅
- (선택) 품사/길이/불용어 필터 → 최종 토큰
- 문서별 토큰 저장과 **전체/카테고리별 빈도표**만 생성 (그래프/행렬 없음)
- 모든 CSV는 `UTF-8-SIG`로 저장


In [1]:

# (필요시) 설치
# !pip -q install konlpy JPype1 pandas numpy


In [2]:

import os, re, json
from collections import Counter
from typing import List, Tuple, Optional

import pandas as pd
import numpy as np

from konlpy.tag import Okt
OKT = Okt()

# ========= 설정 =========
INPUT_CSV  = "naver_blog_all_.csv"   # ← 파일 경로
OUTPUT_DIR = "morph_only_outputs"    # 결과 폴더
ID_COL     = ""                                 # 선택
CATEGORY_COL = ""                               # 선택

POS_KEEP = ("Noun","Adjective","Verb")          # None이면 모든 품사 유지
MIN_TOKEN_LEN = 2
STOPWORDS_EXTRA_PATH = "/mnt/data/stopwords_extra.txt"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========= 불용어/정규식 =========
DEFAULT_STOPWORDS = set([
    "이","그","저","것","거","수","등","때문","때문에","및","그리고","그러나","그래서","또한",
    "으로","로","은","는","이","가","을","를","과","와","하고","보다","에서","에게","에도","에는",
    "이다","아니다","되다","하다","있다","없다","같다","정도","부분","위해","대한",
    "뿐","처럼","같은","듯","듯이","거의","각각","모든","아무","이런","그런","저런","어떤","무슨","등등",
    "부터","까지","만","더","가장","제일","아주","매우","너무","정말","진짜","그냥","혹시","대부분","여러","수준",
    "제품","상품","구성","구입","구매","배송","포장","가격","행사","세트","옵션","용량","맛","향","느낌",
    "사용","효과","후기","리뷰","평가","별점","평점","추천","만족","불만","개선","재구매","성분","브랜드",
    "수량","개","박스","병","캡슐","정","분","알","가루","분말","ml","mg","g","kg","개입","세일",
    "이벤트","증정","사은품","주문","선물","쇼핑","쇼핑백","리뷰수",
    "먹다","먹기","맛있다","좋다","괜찮다","간편하다","간단하다","꾸준하다","편하다","받다","사다","들다",
    "자다","되다","가다","오다","없다","같다","보다","재다","되었다","되어다","요즘",
])

RE_URL     = re.compile(r"https?://\S+|www\.\S+")
RE_EMAIL   = re.compile(r"[\w\.-]+@[\w\.-]+")
RE_MENTION = re.compile(r"[@#][\w\-_]+")
RE_NON_KR  = re.compile(r"[^0-9A-Za-z가-힣ㄱ-ㅎㅏ-ㅣ\s]")

def load_extra_stopwords(path: str) -> set:
    extra = set()
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                w = line.strip()
                if w and not w.startswith("#"):
                    extra.add(w)
    return extra

def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = RE_URL.sub(" ", s)
    s = RE_EMAIL.sub(" ", s)
    s = RE_MENTION.sub(" ", s)
    s = RE_NON_KR.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def detect_text_col(df: pd.DataFrame, candidates=("review_text","리뷰","본문","content","text","댓글","comment","body","post")) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    for c in df.columns:
        if df[c].dtype == "object":
            return c
    raise ValueError("텍스트 컬럼을 찾지 못했습니다.")


In [3]:

# === 데이터 로드 & 컬럼 탐지 ===
try:
    df = pd.read_csv(INPUT_CSV, encoding="utf-8", low_memory=False)
except Exception:
    df = pd.read_csv(INPUT_CSV, encoding="cp949", low_memory=False)

text_col = detect_text_col(df)

id_col = ID_COL if (ID_COL and ID_COL in df.columns) else ("id" if "id" in df.columns else None)
cat_col = CATEGORY_COL if (CATEGORY_COL and CATEGORY_COL in df.columns) else None
if not cat_col:
    for c in ["카테고리","category","Category","cate","분류","대분류","소분류"]:
        if c in df.columns:
            cat_col = c
            break

print("텍스트 컬럼 :", text_col)
print("ID 컬럼     :", id_col)
print("카테고리 컬럼:", cat_col)

df.head(3)


텍스트 컬럼 : keyword
ID 컬럼     : None
카테고리 컬럼: None


,keyword,title,link,description,bloggername,bloggerlink,postdate
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,도시락 100,광주<b>도시락</b>배달 샐러드로 다이어트까지,https://blog.naver.com/wlgoehdehd/224036526901,"처음에는 <b>도시락</b> 직접 싸가기도 했지만 아침마다 정신없고, 배달음식은 맛...",Love Pop Music Talk (L.P.M.T),blog.naver.com/wlgoehdehd,20251010.0
2,도시락 100,마카오 이심 <b>도시락</b>이심 추천 갤럭시 사용법 할인코드,https://blog.naver.com/19840602/224040238336,<b>도시락</b> 이심 eSIM 사용이었어요. 예전엔 유심을 바꿔 끼우느라 분실 ...,꽃맘이간다,blog.naver.com/19840602,20251014.0


In [4]:

# === 형태소 분석(토큰화) ===
stopwords = DEFAULT_STOPWORDS | load_extra_stopwords(STOPWORDS_EXTRA_PATH)

def tokenize_pos(text: str):
    text = clean_text(text)
    if not text:
        return []
    return OKT.pos(text, norm=True, stem=True)

def filter_tokens(pairs):
    toks = []
    for w,p in pairs:
        if POS_KEEP is not None and p not in POS_KEEP:
            continue
        if len(w) < MIN_TOKEN_LEN:
            continue
        if w in stopwords:
            continue
        toks.append(w)
    return toks

tokens_list = []
pos_json_list = []

for s in df[text_col].fillna("").astype(str):
    pairs = tokenize_pos(s)
    toks  = filter_tokens(pairs)
    tokens_list.append(" ".join(toks))
    pos_json_list.append(json.dumps(pairs, ensure_ascii=False))

out_df = pd.DataFrame({
    text_col: df[text_col].astype(str),
    "__tokens__": tokens_list,
    "__pos_json__": pos_json_list
})

if id_col:
    out_df.insert(0, id_col, df[id_col])
if cat_col:
    out_df.insert(1 if id_col else 0, cat_col, df[cat_col])

tokens_path = os.path.join(OUTPUT_DIR, f"tokens_only_{os.path.splitext(os.path.basename(INPUT_CSV))[0]}.csv")
out_df.to_csv(tokens_path, index=False, encoding="utf-8-sig")
print("문서별 토큰 CSV 저장:", tokens_path)

out_df.head(5)


문서별 토큰 CSV 저장: morph_only_outputs\tokens_only_naver_blog_all_.csv


,keyword,__tokens__,__pos_json__
0,nan,,[]
1,도시락 100,도시락,"[[""도시락"", ""Noun""], [""100"", ""Number""]]"
2,도시락 100,도시락,"[[""도시락"", ""Noun""], [""100"", ""Number""]]"
3,도시락 100,도시락,"[[""도시락"", ""Noun""], [""100"", ""Number""]]"
4,도시락 100,도시락,"[[""도시락"", ""Noun""], [""100"", ""Number""]]"


In [5]:

# === 빈도표(전체/카테고리별) ===
all_tokens = []
for row in out_df["__tokens__"]:
    if not isinstance(row, str) or not row:
        continue
    all_tokens.extend(row.split())

freq_df = pd.DataFrame(Counter(all_tokens).most_common(), columns=["token","freq"])
freq_path = os.path.join(OUTPUT_DIR, f"freq_overall_{os.path.splitext(os.path.basename(INPUT_CSV))[0]}.csv")
freq_df.to_csv(freq_path, index=False, encoding="utf-8-sig")
print("전체 빈도 CSV 저장:", freq_path)

if cat_col:
    rows = []
    for cat, s in out_df.groupby(cat_col)["__tokens__"]:
        c = Counter()
        for doc in s.fillna("").astype(str):
            if doc:
                c.update(doc.split())
        for k,v in c.items():
            rows.append({"category": cat, "token": k, "freq": v})
    freq_cat_df = pd.DataFrame(rows).sort_values(["category","freq"], ascending=[True,False])
    freq_cat_path = os.path.join(OUTPUT_DIR, f"freq_by_category_{os.path.splitext(os.path.basename(INPUT_CSV))[0]}.csv")
    freq_cat_df.to_csv(freq_cat_path, index=False, encoding="utf-8-sig")
    print("카테고리별 빈도 CSV 저장:", freq_cat_path)
else:
    print("카테고리 열 없음: per-category 빈도 생략.")

freq_df.head(10)


전체 빈도 CSV 저장: morph_only_outputs\freq_overall_naver_blog_all_.csv
카테고리 열 없음: per-category 빈도 생략.


,token,freq
0,도시락,100
1,양제,100
2,직장,100
3,점심,100
4,건강,100
5,루틴,100
6,마켓,100
7,컬리,100
8,뷰티,100
